# ABSA Reproducibility on Kaggle
Notebook mau de chay lai toan bo thuc nghiem so sanh Transformer vs LSTM/BiLSTM/RNN.
Chi can Run All la se sinh ra bang ket qua tong hop.

In [ ]:
import os
import glob

# Neu ban clone repo thu cong, sua lai bien REPO_DIR cho dung duong dan
REPO_DIR = "/kaggle/working/xai-transformer"
if not os.path.exists(REPO_DIR):
    print("Repo chua co o /kaggle/working.")
    print("Hay chay: !git clone https://github.com/haiyen040602/xai-absa.git /kaggle/working/xai-transformer")
else:
    print("Repo found:", REPO_DIR)

dataset_candidates = glob.glob('/kaggle/input/*/FINAL_CLEANED_CORRECTED_SHUFFLED_DATASET_NO_DUPLICATE.jsonl')
print('Detected datasets:', dataset_candidates[:5])

In [ ]:
%cd /kaggle/working/xai-transformer
!pip install -q -r kaggle/requirements-kaggle.txt

In [ ]:
# Chay toan bo so sanh: Transformer + LSTM + BiLSTM + RNN
!python kaggle/run_absa_repro.py \
  --experiment all \
  --model_name roberta-base \
  --epochs 3 \
  --batch_size 8 \
  --learning_rate 1e-5 \
  --max_length 256 \
  --baseline_epochs 10 \
  --baseline_batch_size 8 \
  --baseline_learning_rate 1e-3 \
  --baseline_max_length 50 \
  --output_dir /kaggle/working/absa_outputs

In [ ]:
import pandas as pd

comparison_path = '/kaggle/working/absa_outputs/comparison_table.csv'
df = pd.read_csv(comparison_path)
df

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.copy()
plot_df['accuracy'] = plot_df['accuracy'].astype(float)
plot_df['macro_f1'] = plot_df['macro_f1'].astype(float)

fig, ax = plt.subplots(figsize=(9, 5))
x = range(len(plot_df))
width = 0.38

ax.bar([i - width / 2 for i in x], plot_df['accuracy'], width=width, label='Accuracy')
ax.bar([i + width / 2 for i in x], plot_df['macro_f1'], width=width, label='Macro F1')

ax.set_xticks(list(x))
ax.set_xticklabels(plot_df['experiment'].tolist(), rotation=0)
ax.set_ylim(0, 1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy vs Macro F1')
ax.legend()

for i, v in enumerate(plot_df['accuracy']):
    ax.text(i - width / 2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
for i, v in enumerate(plot_df['macro_f1']):
    ax.text(i + width / 2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image
from pathlib import Path

cm_paths = {
    'transformer': Path('/kaggle/working/absa_outputs/transformer/confusion_matrix.png'),
    'lstm': Path('/kaggle/working/absa_outputs/lstm/confusion_matrix.png'),
    'bilstm': Path('/kaggle/working/absa_outputs/bilstm/confusion_matrix.png'),
    'rnn': Path('/kaggle/working/absa_outputs/rnn/confusion_matrix.png'),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (name, path) in zip(axes, cm_paths.items()):
    ax.set_title(name.upper())
    if path.exists():
        image = Image.open(path)
        ax.imshow(image)
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'Missing file:
{path}', ha='center', va='center')
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Tuy chon: chi chay rieng 1 baseline
# !python kaggle/run_absa_repro.py --experiment bilstm --baseline_epochs 10 --output_dir /kaggle/working/absa_outputs_bilstm